# 서울시 부동산 2024 — 전처리(판다스) 4회차 해설/정답


**직거래(1) / 중개거래(0)** 예측을 기준으로,  
Split -> Fit -> Transform 흐름의 방식입니다.

- 결측/이상치/범주형 인코딩까지 끝내고
- 마지막에 로지스틱 회귀로 입력이 잘 만들어졌는지만 체크합니다.


In [47]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("2024.csv", encoding="cp949", low_memory=False)

# 실습 속도 조절용(원하면 사용)
# df = df.sample(50_000, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()


(77523, 21)


,접수연도,자치구코드,자치구명,법정동코드,법정동명,지번구분,지번구분명,본번,부번,건물명,...,물건금액(만원),건물면적(㎡),토지면적(㎡),층,권리구분,취소일,건축년도,건물용도,신고구분,신고한 개업공인중개사 시군구명
0,2024,11380,은평구,10200,녹번동,1.0,대지,0086,0125,성원빌라(86-125),...,16400,53.07,25.0000,4.0,NaN,NaN,1991.0,연립다세대,중개거래,서울 은평구
1,2024,11500,강서구,10300,화곡동,1.0,대지,0924,0004,팔팔빌라,...,35000,39.24,27.0000,2.0,NaN,NaN,1986.0,연립다세대,직거래,NaN
2,2024,11560,영등포구,11100,당산동1가,1.0,대지,0143,0000,다솔시티하임,...,30000,29.54,14.0000,5.0,NaN,NaN,2017.0,연립다세대,직거래,NaN
3,2024,11560,영등포구,12800,양평동4가,1.0,대지,0220,0000,해울빌,...,36000,40.41,63.6400,3.0,NaN,NaN,2021.0,오피스텔,중개거래,서울 용산구
4,2024,11410,서대문구,11200,대현동,1.0,대지,0090,0077,유씨유 이대,...,19300,17.31,30.8725,5.0,NaN,NaN,2018.0,오피스텔,중개거래,서울 서대문구


### 문제 1 해설 — 타깃 만들기 + SPLIT
- `신고구분`이 결측인 2건은 학습 타깃을 만들 수 없으니 제거합니다.
- `stratify=y`는 분류에서 거의 기본값!


In [48]:
# 1) 타깃 결측 제거
df_ml = df.dropna(subset=["신고구분"]).copy()

# 2) y / X
y = (df_ml["신고구분"] == "직거래").astype(int)
X = df_ml.drop(columns=["신고구분"])

# 3) SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

print("X_train:", X_train.shape, "X_test:", X_test.shape)


X_train: (62016, 20) X_test: (15505, 20)


### 문제 2 해설 — 계약일 datetime + 파생
- 날짜 파싱은 `fit`이 없는 전처리라서 train/test에 그냥 동일 적용하면 됩니다.
- 다만 `errors='coerce'`를 주면, 이상한 값이 있어도 NaT로 떨어져서 코드가 안 죽습니다.


In [49]:
# 계약일 -> datetime
for _df in [X_train, X_test]:
    _df["계약일_dt"] = pd.to_datetime(
        _df["계약일"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )
    _df["계약월"] = _df["계약일_dt"].dt.month
    _df["계약일자"] = _df["계약일_dt"].dt.day
    _df["계약요일"] = _df["계약일_dt"].dt.dayofweek  # 월=0

X_train[["계약일", "계약일_dt", "계약월", "계약일자", "계약요일"]].head()


,계약일,계약일_dt,계약월,계약일자,계약요일
23984,20240713,2024-07-13,7,13,5
39275,20240604,2024-06-04,6,4,1
6534,20240831,2024-08-31,8,31,5
72549,20240119,2024-01-19,1,19,4
62528,20240311,2024-03-11,3,11,0


### 문제 3 해설 — 결측 플래그
- 결측을 채우더라도, "원래 결측이었다"는 정보는 남겨두는 경우가 많습니다.


In [50]:
flag_cols = ["토지면적(㎡)", "층", "건축년도", "건물명", "신고한 개업공인중개사 시군구명"]

for col in flag_cols:
    X_train[f"{col}_isna"] = X_train[col].isna().astype(int)
    X_test[f"{col}_isna"] = X_test[col].isna().astype(int)

X_train[[c for c in X_train.columns if c.endswith("_isna")]].head()


,토지면적(㎡)_isna,층_isna,건축년도_isna,건물명_isna,신고한 개업공인중개사 시군구명_isna
23984,0,0,0,0,0
39275,0,1,0,1,0
6534,0,0,0,0,0
72549,0,0,0,0,0
62528,0,0,0,0,0


### 문제 4 해설 — `층` 그룹 중앙값으로 결측 채우기
- 중앙값 테이블(`층_med_table`)은 **train에서만** 만들고
- test는 그 테이블을 붙여서 채웁니다.
- 그룹키가 결측인 행은 중앙값 테이블로도 못 채우니, 마지막에 전체 중앙값으로 한 번 더 마무리하면 안전합니다.


In [51]:
group_keys = ["자치구명", "건물용도"]

# train에서만 기준 테이블 만들기
floor_med_table = (
    X_train.groupby(group_keys)["층"]
    .median()
    .reset_index()
    .rename(columns={"층": "층_med"})
)

# train 적용
X_train = X_train.merge(floor_med_table, on=group_keys, how="left")
X_train["층"] = X_train["층"].fillna(X_train["층_med"])
X_train = X_train.drop(columns=["층_med"])

# test 적용 (test에서 groupby 새로 하면 누수!)
X_test = X_test.merge(floor_med_table, on=group_keys, how="left")
X_test["층"] = X_test["층"].fillna(X_test["층_med"])
X_test = X_test.drop(columns=["층_med"])

X_train[["자치구명", "건물용도", "층"]].head()


,자치구명,건물용도,층
0,강동구,아파트,13.0
1,동대문구,단독다가구,NaN
2,강동구,아파트,3.0
3,강서구,연립다세대,4.0
4,양천구,아파트,14.0


### 문제 5 해설 — 나머지 결측 채우기(train 기준값)
- 숫자는 중앙값이 무난하고
- 범주는 최빈값(mode) 또는 `'미상'` 같은 값으로 채웁니다.
- 핵심: 중앙값/최빈값 같은 기준값은 **train에서만** 만들기.


In [52]:
# 1) 토지면적: train 중앙값
land_median = X_train["토지면적(㎡)"].median()
X_train["토지면적(㎡)"] = X_train["토지면적(㎡)"].fillna(land_median)
X_test["토지면적(㎡)"] = X_test["토지면적(㎡)"].fillna(land_median)

# 2) 건축년도: train 중앙값
year_median = X_train["건축년도"].median()
X_train["건축년도"] = X_train["건축년도"].fillna(year_median)
X_test["건축년도"] = X_test["건축년도"].fillna(year_median)

# 3) 지번구분명: train 최빈값
jibun_mode = X_train["지번구분명"].mode(dropna=True)[0]
X_train["지번구분명"] = X_train["지번구분명"].fillna(jibun_mode)
X_test["지번구분명"] = X_test["지번구분명"].fillna(jibun_mode)

# 4) 자치구명: 결측이면 '미상'
X_train["자치구명"] = X_train["자치구명"].fillna("미상")
X_test["자치구명"] = X_test["자치구명"].fillna("미상")

# 5) 건물명 / 중개사구: 결측이면 '미상'
X_train["건물명"] = X_train["건물명"].fillna("미상")
X_test["건물명"] = X_test["건물명"].fillna("미상")

X_train["신고한 개업공인중개사 시군구명"] = X_train["신고한 개업공인중개사 시군구명"].fillna("미상")
X_test["신고한 개업공인중개사 시군구명"] = X_test["신고한 개업공인중개사 시군구명"].fillna("미상")

X_train[["토지면적(㎡)", "건축년도", "지번구분명", "자치구명", "건물명", "신고한 개업공인중개사 시군구명"]].head()


,토지면적(㎡),건축년도,지번구분명,자치구명,건물명,신고한 개업공인중개사 시군구명
0,0.00,2000.0,대지,강동구,선사현대,서울 강동구
1,121.00,2015.0,대지,동대문구,미상,서울 성북구
2,0.00,2020.0,대지,강동구,고덕아르테온,서울 강동구
3,22.29,2011.0,대지,강서구,태영아트빌,서울 강서구
4,0.00,2003.0,대지,양천구,세종그랑시아,서울 양천구


### 문제 6 해설 — 이상치(IQR) 캡핑 + log1p
- 금액/면적은 극단값이 있어서, 캡핑 후 log1p까지 해주면 선형 모델이 훨씬 편해집니다.
- 경계(lo/hi)는 **train에서만** 계산합니다.


In [53]:
# 1) 가격 캡핑
q1, q3 = X_train["물건금액(만원)"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

X_train["가격_cap"] = X_train["물건금액(만원)"].clip(lo, hi)
X_test["가격_cap"] = X_test["물건금액(만원)"].clip(lo, hi)

X_train["가격_log1p"] = np.log1p(X_train["가격_cap"].clip(lower=0))
X_test["가격_log1p"] = np.log1p(X_test["가격_cap"].clip(lower=0))

# 2) 면적 캡핑
q1, q3 = X_train["건물면적(㎡)"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

X_train["면적_cap"] = X_train["건물면적(㎡)"].clip(lo, hi)
X_test["면적_cap"] = X_test["건물면적(㎡)"].clip(lo, hi)

X_train["면적_log1p"] = np.log1p(X_train["면적_cap"].clip(lower=0))
X_test["면적_log1p"] = np.log1p(X_test["면적_cap"].clip(lower=0))

X_train[["물건금액(만원)", "가격_cap", "가격_log1p", "건물면적(㎡)", "면적_cap", "면적_log1p"]].head()


,물건금액(만원),가격_cap,가격_log1p,건물면적(㎡),면적_cap,면적_log1p
0,101000,101000,11.522886,72.85,72.85,4.302036
1,145000,145000,11.884496,225.90,149.40,5.013298
2,135000,135000,11.813037,59.98,59.98,4.110546
3,22000,22000,9.998843,38.72,38.72,3.681855
4,100000,100000,11.512935,111.10,111.10,4.719391


### 문제 7 해설 — 파생변수
- 부동산에서 기본 중 기본: `단가(가격/면적)`
- `건축연령`(연식)은 가격/거래 방식과 관계가 있을 수 있습니다.
- `is_high_floor`는 아주 단순하지만, 의외로 신호가 되는 경우가 있어요.


In [54]:
# 1) price_per_sqm
denom = X_train["면적_cap"].replace(0, np.nan)
X_train["price_per_sqm"] = X_train["가격_cap"] / denom
X_test["price_per_sqm"] = X_test["가격_cap"] / X_test["면적_cap"].replace(0, np.nan)

# 혹시라도 생긴 결측은 0 처리(면적 0은 거의 없지만 안전장치)
X_train["price_per_sqm"] = X_train["price_per_sqm"].fillna(0)
X_test["price_per_sqm"] = X_test["price_per_sqm"].fillna(0)

# 2) 건축연령 = 계약연 - 건축년도
# 계약일_dt가 NaT면 2024로 채워도 OK
X_train["계약연"] = X_train["계약일_dt"].dt.year.fillna(2024).astype(int)
X_test["계약연"] = X_test["계약일_dt"].dt.year.fillna(2024).astype(int)

X_train["건축연령"] = X_train["계약연"] - X_train["건축년도"]
X_test["건축연령"] = X_test["계약연"] - X_test["건축년도"]

# 3) is_high_floor
X_train["is_high_floor"] = (X_train["층"] >= 10).astype(int)
X_test["is_high_floor"] = (X_test["층"] >= 10).astype(int)

# 4) land_ratio
X_train["land_ratio"] = X_train["토지면적(㎡)"] / X_train["면적_cap"].replace(0, np.nan)
X_test["land_ratio"] = X_test["토지면적(㎡)"] / X_test["면적_cap"].replace(0, np.nan)

X_train["land_ratio"] = X_train["land_ratio"].fillna(0)
X_test["land_ratio"] = X_test["land_ratio"].fillna(0)

X_train[["가격_cap", "면적_cap", "price_per_sqm", "건축년도", "계약연", "건축연령", "층", "is_high_floor", "land_ratio"]].head()


,가격_cap,면적_cap,price_per_sqm,건축년도,계약연,건축연령,층,is_high_floor,land_ratio
0,101000,72.85,1386.410432,2000.0,2024,24.0,13.0,1,0.000000
1,145000,149.40,970.548862,2015.0,2024,9.0,NaN,0,0.809906
2,135000,59.98,2250.750250,2020.0,2024,4.0,3.0,0,0.000000
3,22000,38.72,568.181818,2011.0,2024,13.0,4.0,0,0.575671
4,100000,111.10,900.090009,2003.0,2024,21.0,14.0,1,0.000000


### 문제 8 해설 — 희귀 범주 묶기 + 원-핫 + 컬럼 정렬 (+누수 컬럼 제거)
- `법정동명`은 범주 수가 많아서 바로 원-핫하면 컬럼이 폭발합니다.
- train에서 상위 20개만 남기고 나머지는 `'기타'`로 묶어 안정화합니다.
- `get_dummies` 후에는 test를 train 컬럼에 맞춰 `reindex`!

**누수 체크 포인트**
- `신고한 개업공인중개사 시군구명`(그리고 결측 플래그)은 직거래 라벨과 거의 1:1로 맞물려서 점수가 비정상적으로 잘 나올 수 있어요.
- 그래서 이번 베이스라인에서는 해당 컬럼을 **모델 입력에서 제외**합니다.

추가로,
- `건물명`은 거의 ID라서 원-핫하면 과적합/컬럼폭발 위험이 큽니다 -> 보통 drop.


In [55]:
# 1) 법정동명_top (train에서 기준 만들기)
top_dongs = X_train["법정동명"].value_counts().head(20).index
X_train["법정동명_top"] = X_train["법정동명"].where(X_train["법정동명"].isin(top_dongs), "기타")
X_test["법정동명_top"] = X_test["법정동명"].where(X_test["법정동명"].isin(top_dongs), "기타")

# 2) 모델에서 빼도 되는 원본 컬럼 정리 (+ 누수 컬럼 제거)
drop_cols = [
    "접수연도", "자치구코드", "법정동코드",
    "계약일", "계약일_dt", "법정동명",
    "신고한 개업공인중개사 시군구명",
    "신고한 개업공인중개사 시군구명_isna",  # 누수 가능성이 매우 높음
    "지번구분", "본번", "부번",
    "취소일", "권리구분",
    "건물명",  # 고유값 너무 많음
    "물건금액(만원)", "건물면적(㎡)",  # log/cap 버전으로 대체
    "가격_cap", "면적_cap"  # 파생변수 만들었으면 원본은 굳이
]
drop_cols = [c for c in drop_cols if c in X_train.columns]

X_train_model = X_train.drop(columns=drop_cols)
X_test_model = X_test.drop(columns=drop_cols)

cat_cols = ["자치구명", "건물용도", "지번구분명", "법정동명_top"]

X_train_dum = pd.get_dummies(X_train_model, columns=cat_cols, drop_first=True)
X_test_dum = pd.get_dummies(X_test_model, columns=cat_cols, drop_first=True)

# train 컬럼 기준으로 test 맞추기
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)

print("X_train_dum:", X_train_dum.shape)
print("X_test_dum :", X_test_dum.shape)

X_train_dum.head()


X_train_dum: (62016, 66)
X_test_dum : (15505, 66)


,토지면적(㎡),층,건축년도,계약월,계약일자,계약요일,토지면적(㎡)_isna,층_isna,건축년도_isna,건물명_isna,...,법정동명_top_수유동,법정동명_top_신림동,법정동명_top_신월동,법정동명_top_신정동,법정동명_top_응암동,법정동명_top_자양동,법정동명_top_중계동,법정동명_top_창동,법정동명_top_천호동,법정동명_top_화곡동
0,0.00,13.0,2000.0,7,13,5,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,121.00,NaN,2015.0,6,4,1,0,1,0,1,...,False,False,False,False,False,False,False,False,False,False
2,0.00,3.0,2020.0,8,31,5,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,22.29,4.0,2011.0,1,19,4,0,0,0,0,...,False,False,False,False,False,False,False,False,False,True
4,0.00,14.0,2003.0,3,11,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False


### 문제 9 해설 — 스케일링
- 스케일러도 평균/표준편차를 학습하므로 train에서만 fit!


In [56]:
from sklearn.preprocessing import StandardScaler

# 수치형 컬럼만 스케일링 (더미 컬럼은 0/1이라 보통 스케일링 안 해도 됨)
scale_cols = [
    "가격_log1p", "면적_log1p", "price_per_sqm",
    "건축연령", "층", "토지면적(㎡)",
    "계약월", "계약일자", "계약요일"
]
scale_cols = [c for c in scale_cols if c in X_train_dum.columns]

scaler = StandardScaler()
X_train_dum[scale_cols] = scaler.fit_transform(X_train_dum[scale_cols])
X_test_dum[scale_cols] = scaler.transform(X_test_dum[scale_cols])

X_train_dum, X_test_dum


(        토지면적(㎡)         층    건축년도       계약월      계약일자      계약요일  토지면적(㎡)_isna  \
 0     -0.133819  0.806787  2000.0  0.569769 -0.335424  1.148367             0   
 1      0.789891       NaN  2015.0  0.175759 -1.354116 -1.007604             0   
 2     -0.133819 -0.775431  2020.0  0.963778  1.701959  1.148367             0   
 3      0.036342 -0.617209  2011.0 -1.794288  0.343703  0.609374             0   
 4     -0.133819  0.965008  2003.0 -1.006269 -0.561800 -1.546597             0   
 ...         ...       ...     ...       ...       ...       ...           ...   
 62011  0.200472 -0.300766  2004.0  1.357788  0.343703  0.070381             0   
 62012  0.087948 -0.775431  2002.0 -0.218250 -1.014552 -1.007604             0   
 62013 -0.133819  0.806787  2005.0  0.569769 -0.788176 -1.007604             0   
 62014  0.236963 -0.300766  2009.0  0.569769 -0.335424  1.148367             0   
 62015  0.095200 -0.775431  2023.0  0.963778  1.362395 -0.468611             0   
 
        층_isna